In [42]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import collections

# Загружаем данные
with open('../data/train.jsonl', encoding='utf-8') as f:
    data = [json.loads(l) for l in f if l.strip()]

df = pd.DataFrame(data)
print(f"Total rows: {len(df)}")
print(f"Unique inputs: {df['input'].nunique()}")
print(f"Unique outputs: {df['output'].nunique()}")

# Длины
df['input_len'] = df['input'].str.len()
df['output_len'] = df['output'].str.len()

print(f"\nInput length: min={df['input_len'].min()}, max={df['input_len'].max()}, mean={df['input_len'].mean():.1f}")
print(f"Output length: min={df['output_len'].min()}, max={df['output_len'].max()}, mean={df['output_len'].mean():.1f}")

# Buckets
df['output_bucket'] = pd.cut(df['output_len'], bins=[0, 20, 50, 100, 200, 1000], labels=['<20', '20-50', '50-100', '100-200', '200+'])
print("\nOutput length buckets:")
print(df['output_bucket'].value_counts().sort_index())

# Показываем первые 5
df[['input', 'output']].head()

Total rows: 15208
Unique inputs: 11450
Unique outputs: 2985

Input length: min=2, max=186, mean=18.8
Output length: min=1, max=146, mean=21.5

Output length buckets:
output_bucket
<20        9362
20-50      4850
50-100      948
100-200      48
200+          0
Name: count, dtype: int64


,input,output
0,1/n + 1/m,\frac{1}{n} + \frac{1}{m}
1,1/n+1/m,\frac{1}{n} + \frac{1}{m}
2,1 делить на n + 1 делить на m,\frac{1}{n} + \frac{1}{m}
3,one over n plus one over m,\frac{1}{n} + \frac{1}{m}
4,1/N+1/M,\frac{1}{n} + \frac{1}{m}


In [43]:
print("\nTop 10 outputs:")
print(df['output'].value_counts().head(10))
df_unique = df.drop_duplicates(subset=['input'], keep='first')
print(f"After dedup: {len(df_unique)}")


Top 10 outputs:
output
\lim_{x \to 0} \frac{\sin(x)}{x} = 1                      44
\lim_{x \to 0} \frac{\ln(1+x)}{x} = 1                     39
\lim_{x \to 0} \frac{\tan(x)}{x} = 1                      34
\lim_{x \to 0} \frac{e^{x} - 1}{x} = 1                    30
\lim_{x \to 0} \frac{1 - \cos(x)}{x^{2}} = \frac{1}{2}    30
a^{2} + b^{2} = c^{2}                                     29
\lim_{x \to 0} (1 + x)^{1/x} = e                          25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n} = e             25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n+1} = e           25
\lim_{x \to \infty} (1 + \frac{a}{x})^{x} = e^{a}         25
Name: count, dtype: int64
After dedup: 11450


In [44]:
# Топ-20 outputs
print("Top 20 outputs:")
print(df['output'].value_counts().head(20))

# Топ-20 inputs
print("\nTop 20 inputs:")
print(df['input'].value_counts().head(20))

# Input length distribution
df['input_bucket'] = pd.cut(df['input_len'], bins=[0, 20, 50, 100, 200], labels=['<20', '20-50', '50-100', '100-200'])
print("\nInput length buckets:")
print(df['input_bucket'].value_counts().sort_index())

Top 20 outputs:
output
\lim_{x \to 0} \frac{\sin(x)}{x} = 1                                      44
\lim_{x \to 0} \frac{\ln(1+x)}{x} = 1                                     39
\lim_{x \to 0} \frac{\tan(x)}{x} = 1                                      34
\lim_{x \to 0} \frac{e^{x} - 1}{x} = 1                                    30
\lim_{x \to 0} \frac{1 - \cos(x)}{x^{2}} = \frac{1}{2}                    30
a^{2} + b^{2} = c^{2}                                                     29
\lim_{x \to 0} (1 + x)^{1/x} = e                                          25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n} = e                             25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n+1} = e                           25
\lim_{x \to \infty} (1 + \frac{a}{x})^{x} = e^{a}                         25
\lim_{x \to \infty} (1 + \frac{a}{x})^{bx} = e^{ab}                       25
\lim_{x \to 0} (1 + ax)^{b/x} = e^{ab}                                    25
\lim_{x \to 0} (1 + x)^{a/x} = e^{a}                 

In [45]:
# Оставляем по 5 примеров на каждый уникальный output
df_balanced = df.groupby('output').head(5).reset_index(drop=True)

In [46]:
def categorize(output):
    if '\\lim' in output: return 'limit'
    if '\\int' in output: return 'integral'
    if '\\sum' in output: return 'sum'
    if '\\frac{d}{dx}' in output or "f'" in output: return 'derivative'
    if '\\sqrt' in output: return 'sqrt'
    if '\\log' in output or '\\ln' in output: return 'log'
    if '\\sin' in output or '\\cos' in output or '\\tan' in output: return 'trig'
    if '\\frac' in output: return 'fraction'
    if '=' in output: return 'equation'
    if '\\alpha' in output or '\\beta' in output or '\\gamma' in output or '\\theta' in output: return 'greek'
    if '^' in output: return 'power'
    return 'simple'

df['category'] = df['output'].apply(categorize)

# Проверка
print(df['category'].value_counts())

category
fraction      2580
equation      2080
power         1860
simple        1779
limit         1466
trig          1116
sqrt           943
sum            805
integral       691
derivative     687
log            651
greek          550
Name: count, dtype: int64


In [47]:
MAX_PER_CATEGORY = 1500

# Группируем и сэмплируем вручную
sampled_dfs = []
for cat, group in df.groupby('category'):
    if len(group) > MAX_PER_CATEGORY:
        sampled = group.sample(MAX_PER_CATEGORY, random_state=42)
    else:
        sampled = group
    sampled_dfs.append(sampled)

df_balanced = pd.concat(sampled_dfs, ignore_index=True)

print(f"After downsampling: {len(df_balanced)}")
print(f"Columns: {df_balanced.columns.tolist()}")
print("\nCategory distribution:")
print(df_balanced['category'].value_counts())
print("\nPercentages:")
print((df_balanced['category'].value_counts(normalize=True) * 100).round(1))

print(f"After downsampling: {len(df_balanced)}")
print(f"Columns: {df_balanced.columns.tolist()}")
print("\nCategory distribution:")
print(df_balanced['category'].value_counts())
print("\nPercentages:")
print((df_balanced['category'].value_counts(normalize=True) * 100).round(1))

After downsampling: 12909
Columns: ['input', 'output', 'input_len', 'output_len', 'output_bucket', 'input_bucket', 'category']

Category distribution:
category
equation      1500
fraction      1500
power         1500
simple        1500
limit         1466
trig          1116
sqrt           943
sum            805
integral       691
derivative     687
log            651
greek          550
Name: count, dtype: int64

Percentages:
category
equation      11.6
fraction      11.6
power         11.6
simple        11.6
limit         11.4
trig           8.6
sqrt           7.3
sum            6.2
integral       5.4
derivative     5.3
log            5.0
greek          4.3
Name: proportion, dtype: float64
After downsampling: 12909
Columns: ['input', 'output', 'input_len', 'output_len', 'output_bucket', 'input_bucket', 'category']

Category distribution:
category
equation      1500
fraction      1500
power         1500
simple        1500
limit         1466
trig          1116
sqrt           943
sum      

In [49]:
unique_inputs = df_balanced['input'].nunique()
print(f"Unique inputs: {unique_inputs} / {len(df_balanced)}")
print(f"Duplicates: {len(df_balanced) - unique_inputs}")

Unique inputs: 9945 / 12909
Duplicates: 2964


In [50]:
# Дедупликация
df_unique = df_balanced.drop_duplicates(subset=['input'], keep='first').reset_index(drop=True)

print(f"After dedup: {len(df_unique)}")
print(f"Removed: {len(df_balanced) - len(df_unique)}")
print("\nCategory distribution:")
print(df_unique['category'].value_counts())
print("\nPercentages:")
print((df_unique['category'].value_counts(normalize=True) * 100).round(1))

After dedup: 9945
Removed: 2964

Category distribution:
category
power         1290
fraction      1284
simple        1264
equation      1207
trig           775
sqrt           726
limit          683
sum            657
derivative     614
log            541
integral       505
greek          399
Name: count, dtype: int64

Percentages:
category
power         13.0
fraction      12.9
simple        12.7
equation      12.1
trig           7.8
sqrt           7.3
limit          6.9
sum            6.6
derivative     6.2
log            5.4
integral       5.1
greek          4.0
Name: proportion, dtype: float64


In [51]:
with open('../data/train_balanced.jsonl', 'w', encoding='utf-8') as f:
    for _, row in df_unique.iterrows():
        f.write(json.dumps({'input': row['input'], 'output': row['output']}, ensure_ascii=False) + '\n')

print(f"Saved: {len(df_unique)} rows")
print(f"Path: ../data/train_balanced.jsonl")

Saved: 9945 rows
Path: ../data/train_balanced.jsonl
